# Comparativa entre variants del model

Llegeix tots els CSVs disponibles a `outputs/` i compara:

- **Diversitat lèxica** (vocab, captions úniques)
- **Mètriques globals** (BLEU-1/4, CLIPScore)
- **Biaix de gènere** amb **intervals de confiança bootstrap (95%)**
- **Decoding** beam vs nucleus sampling

Posa qualsevol subconjunt d'aquests CSVs a `../outputs/`:
- `predictions_zeroshot.csv` — pretrained sense fine-tuning
- `predictions.csv` (o `predictions_v1.csv`) — baseline fine-tuned, sense debias
- `predictions_v2.csv` — debias 1.8×
- `predictions_v3.csv` — debias 2.5×
- `predictions_v3_nucleus.csv` — v3 amb nucleus sampling

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import matplotlib.pyplot as plt

OUTPUTS_DIR = '../outputs'

# Map of label -> filename. Add/remove as needed.
VARIANTS = [
    ('zero-shot',       'predictions_zeroshot.csv'),
    ('v1 baseline',     'predictions.csv'),
    ('v2 debias 1.8x',  'predictions_v2.csv'),
    ('v3 debias 2.5x',  'predictions_v3.csv'),
    ('v3 + nucleus',    'predictions_v3_nucleus.csv'),
]

dfs = {}
for label, fname in VARIANTS:
    path = os.path.join(OUTPUTS_DIR, fname)
    if os.path.exists(path):
        dfs[label] = pd.read_csv(path)
        print(f'  {label:18s}  {fname:35s}  rows={len(dfs[label])}')
    else:
        print(f'  (skip) {label:18s}  {fname} not found')

## Mètriques agregades per variant

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z']+", str(text).lower())

# Corpus BLEU per dataframe (alineat amb train.py:evaluate)
_smooth = SmoothingFunction().method1

def corpus_bleu_df(df, n):
    hyps = [tokenize(p) for p in df['predicted_caption']]
    refs = [[tokenize(r) for r in str(rs).split(' ||| ')] for rs in df['references']]
    weights = tuple([1/n]*n + [0]*(4-n))
    return corpus_bleu(refs, hyps, weights=weights, smoothing_function=_smooth)

rows = []
for label, df in dfs.items():
    all_toks = [t for c in df['predicted_caption'] for t in tokenize(c)]
    rows.append({
        'variant':         label,
        'n':               len(df),
        'BLEU-1 (corpus)': corpus_bleu_df(df, 1),
        'BLEU-4 (corpus)': corpus_bleu_df(df, 4),
        'CLIPScore':       df['clip_score'].mean(),
        'pred_vocab':      len(set(all_toks)),
        'unique_captions': df['predicted_caption'].nunique(),
        'pred_avg_len':    df['pred_length'].mean(),
    })
summary = pd.DataFrame(rows).set_index('variant')
summary.round({'BLEU-1 (corpus)': 4, 'BLEU-4 (corpus)': 4, 'CLIPScore': 2, 'pred_avg_len': 2})

## Biaix de gènere amb intervals de confiança bootstrap

Per a cada variant, calculem:
- `woman→man %` — % de refs amb dona que el model prediu home
- `man→woman %` — el simètric
- `asimetria` — quocient dels dos (1.0 = sense biaix)

Intervals de confiança calculats per **bootstrap (1000 resamples) percentil 95%**.

In [ ]:
def gender_label(row, ref=True):
    m = row['refs_mention_man'] if ref else row['pred_mentions_man']
    w = row['refs_mention_woman'] if ref else row['pred_mentions_woman']
    if m and not w: return 'man'
    if w and not m: return 'woman'
    if m and w:     return 'both'
    return 'neutral'

def add_gender_cols(df):
    df = df.copy()
    df['ref_gender']  = df.apply(lambda r: gender_label(r, ref=True),  axis=1)
    df['pred_gender'] = df.apply(lambda r: gender_label(r, ref=False), axis=1)
    return df

In [ ]:
def swap_rates(df):
    """Return (woman→man rate, man→woman rate). Each rate = swaps / refs_of_that_gender."""
    n_w_refs = (df['ref_gender'] == 'woman').sum()
    n_m_refs = (df['ref_gender'] == 'man').sum()
    w_to_m = ((df['ref_gender'] == 'woman') & (df['pred_gender'] == 'man')).sum()
    m_to_w = ((df['ref_gender'] == 'man')   & (df['pred_gender'] == 'woman')).sum()
    r_wm = w_to_m / n_w_refs if n_w_refs else np.nan
    r_mw = m_to_w / n_m_refs if n_m_refs else np.nan
    return r_wm, r_mw

def bootstrap_ci(df, fn, n_iter=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n = len(df)
    samples = []
    for _ in range(n_iter):
        idx = rng.integers(0, n, n)
        samples.append(fn(df.iloc[idx]))
    arr = np.array(samples)
    lo = np.percentile(arr, 100 * alpha / 2, axis=0)
    hi = np.percentile(arr, 100 * (1 - alpha / 2), axis=0)
    return lo, hi

In [ ]:
rows = []
gender_dfs = {label: add_gender_cols(df) for label, df in dfs.items()}

for label, df in gender_dfs.items():
    r_wm, r_mw = swap_rates(df)
    asym = r_wm / r_mw if r_mw and not np.isnan(r_mw) and r_mw > 0 else np.nan

    def stats_fn(d):
        a, b = swap_rates(d)
        c = a / b if b and not np.isnan(b) and b > 0 else np.nan
        return [a, b, c]
    lo, hi = bootstrap_ci(df, stats_fn, n_iter=1000)

    rows.append({
        'variant': label,
        'woman→man %': f'{100*r_wm:.1f} [{100*lo[0]:.1f}, {100*hi[0]:.1f}]',
        'man→woman %': f'{100*r_mw:.1f} [{100*lo[1]:.1f}, {100*hi[1]:.1f}]',
        'asimetria': f'{asym:.2f} [{lo[2]:.2f}, {hi[2]:.2f}]',
        '_wm': r_wm, '_mw': r_mw, '_asym': asym,
        '_wm_lo': lo[0], '_wm_hi': hi[0],
        '_mw_lo': lo[1], '_mw_hi': hi[1],
        '_asym_lo': lo[2], '_asym_hi': hi[2],
    })

gender_summary = pd.DataFrame(rows).set_index('variant')
gender_summary[['woman→man %', 'man→woman %', 'asimetria']]

In [ ]:
# Plot: swap rates with error bars
labels = list(gender_summary.index)
x = np.arange(len(labels))
wm = gender_summary['_wm'].values * 100
mw = gender_summary['_mw'].values * 100
wm_err = np.array([(gender_summary['_wm'] - gender_summary['_wm_lo']).values,
                   (gender_summary['_wm_hi'] - gender_summary['_wm']).values]) * 100
mw_err = np.array([(gender_summary['_mw'] - gender_summary['_mw_lo']).values,
                   (gender_summary['_mw_hi'] - gender_summary['_mw']).values]) * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, wm, 0.4, label='woman→man %', yerr=wm_err, capsize=4, color='#d62728')
ax.bar(x + 0.2, mw, 0.4, label='man→woman %', yerr=mw_err, capsize=4, color='#1f77b4')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel('% de swap (95% CI bootstrap)')
ax.set_title('Biaix de gènere per variant')
ax.axhline(10, color='gray', linestyle='--', alpha=0.5, label='objectiu <10%')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Plot: asymmetry ratio with error bars
asym = gender_summary['_asym'].values
asym_err = np.array([(gender_summary['_asym'] - gender_summary['_asym_lo']).values,
                     (gender_summary['_asym_hi'] - gender_summary['_asym']).values])

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ca02c' if abs(a - 1) < (h - a) else '#d62728'
          for a, h in zip(asym, gender_summary['_asym_hi'].values)]
ax.bar(labels, asym, yerr=asym_err, capsize=4, color=colors)
ax.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='simetria perfecta')
ax.set_ylabel('asimetria (woman→man / man→woman)')
ax.set_title('Asimetria de gènere (verd: CI inclou 1.0)')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## Casos arreglats vs residuals després del debiasing

Per a les imatges on una variant prèvia (v1 o zero-shot) prediu "home" sobre una referència femenina, mira si v3 ho ha corregit.
Mostra la matriu de prediccions de totes les variants i separa **arreglats** (v3 ja no prediu home) de **residuals** (v3 encara prediu home).


In [ ]:
# Identifica casos on una variant prèvia (v1 o zero-shot) prediu home/noi però v3 ja no.
# Mostra TOTES les imatges (no només head(8)) i destaca les arreglades.

import re

MAL_KEYWORDS = [' man ', ' men ', ' boy', ' boys', ' male', ' gentleman', ' father', ' he ', ' his ']
FEM_KEYWORDS = [' woman', ' women', ' girl', ' girls', ' lady', ' ladies', ' female', ' mother', ' she ', ' her ']

def gender_of(text):
    t = ' ' + str(text).lower() + ' '
    f = any(k in t for k in FEM_KEYWORDS)
    m = any(k in t for k in MAL_KEYWORDS)
    if f and not m: return 'woman'
    if m and not f: return 'man'
    if f and m:     return 'both'
    return 'neutral'

# Pren la variant més "antiga" disponible com a referència del biaix original
prior_label = None
for cand in ['v1 baseline', 'v1', 'zero-shot']:
    if cand in gender_dfs:
        prior_label = cand
        break

if prior_label is None or 'v3 debias 2.5x' not in gender_dfs:
    print('Cal almenys una variant prèvia (v1 o zero-shot) i v3 debias 2.5x als CSVs.')
else:
    prior = gender_dfs[prior_label]
    v3 = gender_dfs['v3 debias 2.5x']
    
    # Tots els swaps dona→home a la variant prèvia
    swap_imgs = prior[(prior['ref_gender'] == 'woman') & (prior['pred_gender'] == 'man')]['image_name'].drop_duplicates().tolist()
    print(f'Swap dona→home a {prior_label}: {len(swap_imgs)} imatges úniques')
    
    rows = []
    for img in swap_imgs:
        row = {'image': img}
        for label, df in dfs.items():
            match = df[df['image_name'] == img]
            row[label] = match['predicted_caption'].iloc[0] if len(match) else '—'
        # Estat a v3: arreglat si v3 ja no prediu home
        v3_pred = row.get('v3 debias 2.5x', '')
        row['v3_status'] = 'FIXED ✓' if gender_of(v3_pred) != 'man' else 'residual ✗'
        rows.append(row)
    
    if not rows:
        print('Cap swap a la variant prèvia.')
    else:
        comp = pd.DataFrame(rows).set_index('image')
        pd.set_option('display.max_colwidth', 80)
        
        fixed = comp[comp['v3_status'] == 'FIXED ✓']
        residual = comp[comp['v3_status'] == 'residual ✗']
        print(f'\n→ {len(fixed)} arreglats per v3, {len(residual)} residuals\n')
        
        print('=== ARREGLATS PER v3 ===')
        display(fixed.drop(columns=['v3_status']))
        
        print('\n=== RESIDUALS (v3 encara prediu home) ===')
        display(residual.drop(columns=['v3_status']))


## Visualització dels casos arreglats

Reutilitza `show_examples()` (definida originalment a `bias_analysis.ipynb`) per ensenyar **3 imatges on la variant prèvia equivocava el gènere i v3 ho ha corregit**.

El que es mostra com a `Pred:` és la **predicció INCORRECTA de la variant prèvia** — perquè el contrast amb la referència sigui visible.


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

IMAGES_DIR = '../data/flickr8k/Images'

def show_examples(subset, title, n=6):
    print(f'\n=== {title} ===')
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (_, row) in zip(axes, subset.head(n).iterrows()):
        path = os.path.join(IMAGES_DIR, row['image_name'])
        try:
            ax.imshow(Image.open(path))
        except Exception as e:
            ax.text(0.5, 0.5, f'(no img)\n{e}', ha='center')
        ax.axis('off')
        title_txt = f"BLEU-1={row['bleu_1']:.2f} CLIP={row['clip_score']:.1f}\nPred: {row['predicted_caption'][:60]}"
        ax.set_title(title_txt, fontsize=8)
    plt.tight_layout(); plt.show()
    for _, row in subset.head(n).iterrows():
        print(f"[{row['image_name']}]")
        print(f"  Pred (incorrecta): {row['predicted_caption']}")
        print(f"  Refs:              {row['references'][:200]}")

# Tria la variant prèvia disponible i v3
prior_label = next((c for c in ['v1 baseline', 'v1', 'zero-shot'] if c in gender_dfs), None)
if prior_label is None or 'v3 debias 2.5x' not in gender_dfs:
    print('Cal una variant prèvia (v1 o zero-shot) i v3 debias 2.5x als CSVs.')
else:
    prior = gender_dfs[prior_label]
    v3 = gender_dfs['v3 debias 2.5x']

    # Imatges on prior va swap dona→home
    prior_swaps = prior[(prior['ref_gender'] == 'woman') & (prior['pred_gender'] == 'man')].drop_duplicates('image_name')

    # Filtra les arreglades a v3 (v3 ja no prediu home per a la mateixa imatge)
    v3_by_img = v3.drop_duplicates('image_name').set_index('image_name')['pred_gender']
    fixed_mask = prior_swaps['image_name'].map(v3_by_img).fillna('man') != 'man'
    fixed = prior_swaps[fixed_mask]

    print(f'Casos arreglats: {len(fixed)} (de {len(prior_swaps)} swaps a {prior_label})')

    n_show = min(3, len(fixed))
    if n_show > 0:
        show_examples(fixed, f'{n_show} casos arreglats — predicció INCORRECTA de {prior_label}', n=n_show)
    else:
        print('Cap cas arreglat trobat amb aquesta combinació de CSVs.')


## Decoding: beam vs nucleus (només si tens v3_nucleus)

In [ ]:
if 'v3 + nucleus' in dfs and 'v3 debias 2.5x' in dfs:
    v3 = dfs['v3 debias 2.5x']
    v3n = dfs['v3 + nucleus']

    def stats(df):
        all_toks = [t for c in df['predicted_caption'] for t in tokenize(c)]
        return {
            'BLEU-1 (corpus)': corpus_bleu_df(df, 1),
            'BLEU-4 (corpus)': corpus_bleu_df(df, 4),
            'CLIPScore': df['clip_score'].mean(),
            'vocab': len(set(all_toks)),
            'unique captions %': 100 * df['predicted_caption'].nunique() / len(df),
            'avg length': df['pred_length'].mean(),
        }
    comp = pd.DataFrame({'beam (v3)': stats(v3), 'nucleus (v3)': stats(v3n)})
    comp['Δ'] = comp['nucleus (v3)'] - comp['beam (v3)']
    print(comp.round(4))

    # Show some side-by-side examples
    common = v3.merge(v3n, on='image_name', suffixes=('_beam', '_nucleus'))
    diffs = common[common['predicted_caption_beam'] != common['predicted_caption_nucleus']].head(8)
    print('\nExemples on canvia la predicció:')
    for _, row in diffs.iterrows():
        print(f"\n[{row['image_name']}]")
        print(f"  beam   : {row['predicted_caption_beam']}")
        print(f"  nucleus: {row['predicted_caption_nucleus']}")
else:
    print('Need both v3 and v3_nucleus CSVs for this comparison.')

## Conclusions per al report

Plantilla a completar amb les xifres dels gràfics:

**Trajectòria del biaix de gènere:**
- Zero-shot: __% / __% → ___× asimetria
- v1 baseline: __% / __% → ___×
- v3 debias 2.5×: __% / __% → ___×

**Lectura crítica:**
- Si zero-shot < v1 → el fine-tuning **va amplificar** el biaix existent al pretrenat.
- Si v3 CI inclou 1.0 → la simetria és **estadísticament indistingible** de perfecta.
- Si nucleus puja vocab i CLIP es manté → decoding alternatiu **recomanable**.

**Limitacions:**
- N=234 mostres femenines → CI ample, ens cal precaució en afirmacions fines.
- El biaix residual (≈12%) pot venir de mostres genuïnament ambigües (cos parcial, distància, ocultació).
- Només es mira gènere binari amb keywords; un treball més complet usaria detectors visuals.